In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只在长度 ≤ 8、维度 ≤ 64 的玩具张量上做矩阵乘法，全程 CPU 秒级跑完，
# 所以不强制 GPU，只打印环境信息确认 PyTorch 可用。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# torch:       张量运算 + F.scaled_dot_product_attention——Colab 默认已装且够新，
#              故意【不】加 -U：会话中途升级 torch 容易让内核半新半旧后续 import 报错。
# matplotlib:  画注意力权重热力图、缩放前后 softmax 分布的对比柱状图。
!pip install -q -U matplotlib

In [ ]:
# ============================================================
# Cell 2: 手算最小例子——逐步走完打分→缩放→softmax→加权求和（对应第 3.3/4.1/4.2 节）
# ============================================================
# 直接给定 Q/K/V（跳过 X→QKV 的投影），用 L=2、d_k=2 的玩具尺寸，
# 把 md 第 3-4 节的手算结果一字不差复现出来。
import torch
import torch.nn.functional as F

Q = torch.tensor([[1., 0.],
                  [0., 1.]])          # [L=2, d_k=2] 两个 query
K = torch.tensor([[1., 0.],
                  [1., 1.]])          # [L=2, d_k=2] 两个 key
V = torch.tensor([[2., 0.],
                  [0., 4.]])          # [L=2, d_v=2] 两个 value
d_k = Q.size(-1)

S = Q @ K.T                            # ① 打分：QKᵀ -> [L, L]
S_scaled = S / (d_k ** 0.5)            # ② 缩放：除以 √d_k
A = F.softmax(S_scaled, dim=-1)        # ③ 逐行 softmax（dim=-1 即沿每一行）-> 注意力权重
O = A @ V                              # ④ 加权求和：AV -> [L, d_v]

print("① 分数 S = QKᵀ:\n", S)
print("② 缩放后 S/√d_k:\n", S_scaled.round(decimals=3))
print("③ 注意力权重 A = softmax(S/√d_k):\n", A.round(decimals=3))
print("   每行之和（应都为 1）:", A.sum(dim=-1).round(decimals=3).tolist())
print("④ 输出 O = A V:\n", O.round(decimals=3))
print("→ 对照 md：A ≈ [[0.5,0.5],[0.33,0.67]]，O ≈ [[1.0,2.0],[0.66,2.68]]")

In [ ]:
# ============================================================
# Cell 3: 封装 scaled_dot_product_attention 函数（对应第 4.3 节）
# ============================================================
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """缩放点积注意力。Q,K,V 形状 [..., L, d_k]（... 是任意前置维，如 batch、head）。
    Attention(Q,K,V) = softmax(QKᵀ / √d_k + mask) V
    参数:
      Q, K, V : [..., L, d_k]（这里取 d_v = d_k）
      mask    : 可选，[..., L, L] 的布尔张量，True 表示该位置【要屏蔽】（置 -inf）
    返回:
      out     : [..., L, d_k] 注意力输出
      attn    : [..., L, L]   注意力权重（已 softmax，便于可视化）
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)      # ① QKᵀ 再 ②缩放 -> [..., L, L]
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf")) # 要屏蔽的位置打 -inf
    attn = F.softmax(scores, dim=-1)                     # ③ 逐行 softmax -> 权重
    out = attn @ V                                       # ④ 加权求和 -> [..., L, d_k]
    return out, attn

# 用 Cell 2 的玩具数据验证封装版和手算一致
Q = torch.tensor([[1., 0.], [0., 1.]])
K = torch.tensor([[1., 0.], [1., 1.]])
V = torch.tensor([[2., 0.], [0., 4.]])
out, attn = scaled_dot_product_attention(Q, K, V)
print("封装版 attn:\n", attn.round(decimals=3))
print("封装版 out :\n", out.round(decimals=3))
print("→ 与 Cell 2 手算结果一致")

In [ ]:
# ============================================================
# Cell 4: 和 PyTorch 官方 F.scaled_dot_product_attention 对齐（对应第 4.3 节）
# ============================================================
# PyTorch 2.0+ 内置了 scaled_dot_product_attention（底层可调用 FlashAttention）。
# 用一组随机的、带 batch+head 维的张量，对比我们手写版与官方版是否数值一致。
torch.manual_seed(0)
B, Hd, L, d_k = 2, 4, 6, 16            # batch=2, head=4, 序列长 6, 每头维度 16
Q = torch.randn(B, Hd, L, d_k)
K = torch.randn(B, Hd, L, d_k)
V = torch.randn(B, Hd, L, d_k)

out_mine, _ = scaled_dot_product_attention(Q, K, V)   # 我们的实现
out_ref = F.scaled_dot_product_attention(Q, K, V)     # PyTorch 官方（默认无 mask、自动缩放）

print("我们的输出形状:", tuple(out_mine.shape))
print("官方输出形状  :", tuple(out_ref.shape))
print("最大绝对误差  :", (out_mine - out_ref).abs().max().item())
print("→ 误差在 1e-6 量级即视为一致（浮点舍入），说明手写实现正确")
assert torch.allclose(out_mine, out_ref, atol=1e-5), "与官方实现不一致！"
print("✅ 通过：手写实现与 PyTorch 官方一致")

In [ ]:
# ============================================================
# Cell 5: 缩放的作用——不除以 √d_k，softmax 会过饱和塌成 one-hot（对应第 3.2 节）
# ============================================================
import matplotlib.pyplot as plt

torch.manual_seed(0)
d_k = 128                              # 故意取较大维度，放大「点积方差随维度涨」的效应
q = torch.randn(d_k)                   # 一个 query
Ks = torch.randn(20, d_k)             # 20 个 key，每个 d_k 维，各维 iid N(0,1)

raw = Ks @ q                           # 不缩放的点积分数 [20]
scaled = raw / (d_k ** 0.5)            # 缩放后的分数

print(f"d_k = {d_k}")
print(f"不缩放分数的标准差: {raw.std().item():.2f}  (理论 ≈ √d_k = {d_k**0.5:.2f})")
print(f"缩放后分数的标准差: {scaled.std().item():.2f}  (理论 ≈ 1)")

a_raw = F.softmax(raw, dim=-1)
a_scaled = F.softmax(scaled, dim=-1)
print(f"不缩放 softmax 的最大权重: {a_raw.max().item():.3f}  (越接近 1 越饱和/塌成 one-hot)")
print(f"缩放后 softmax 的最大权重: {a_scaled.max().item():.3f}  (温和、多个位置都有份)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(range(20), a_raw.numpy(), color="#dc2626")
ax[0].set_title(f"WITHOUT scaling (d_k={d_k}): peaky / near one-hot")
ax[0].set_xlabel("key index"); ax[0].set_ylabel("attention weight"); ax[0].set_ylim(0, 1)
ax[1].bar(range(20), a_scaled.numpy(), color="#2563eb")
ax[1].set_title(f"WITH /sqrt(d_k): smooth distribution")
ax[1].set_xlabel("key index"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()
print("→ 不缩放时某个权重逼近 1、其余≈0（softmax 饱和，梯度几乎为 0，难训练）；")
print("  除以 √d_k 后分布平滑，注意力才能『软』地分给多个位置。")

In [ ]:
# ============================================================
# Cell 6: causal mask——把上三角置 -inf，注意力矩阵变下三角（对应第 5 节）
# ============================================================
torch.manual_seed(1)
L, d_k = 6, 16
Q = torch.randn(L, d_k)
K = torch.randn(L, d_k)
V = torch.randn(L, d_k)

# causal mask：[L, L] 布尔矩阵，上三角（j > i，不含对角线）为 True = 要屏蔽（看不到未来）
causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
print("causal mask（True=屏蔽/未来位置）:\n", causal.int())

_, attn_full = scaled_dot_product_attention(Q, K, V)                 # 不加 mask：全连接
_, attn_causal = scaled_dot_product_attention(Q, K, V, mask=causal)  # 加 causal mask

print("\n不加 mask 的注意力权重（每行和为 1，上三角有值=看到了未来）:\n",
      attn_full.round(decimals=2))
print("\n加 causal mask 后（上三角全 0，每行只看自己及之前）:\n",
      attn_causal.round(decimals=2))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
im0 = ax[0].imshow(attn_full.numpy(), cmap="viridis", vmin=0, vmax=1)
ax[0].set_title("full attention (sees future)")
ax[0].set_xlabel("key j"); ax[0].set_ylabel("query i")
im1 = ax[1].imshow(attn_causal.numpy(), cmap="viridis", vmin=0, vmax=1)
ax[1].set_title("causal mask (lower-triangular)")
ax[1].set_xlabel("key j"); ax[1].set_ylabel("query i")
fig.colorbar(im1, ax=ax, fraction=0.046, label="attention weight")
plt.show()
print("→ 右图上三角全黑（权重=0）：每个 query 只能注意到自己及之前的位置，")
print("  这正是自回归语言模型『预测下一个 token 时不许偷看答案』的实现。")

In [ ]:
# ============================================================
# Cell 7: 最小 self-attention 模块——把投影 + 注意力拼起来（对应第 2 节）
# ============================================================
import torch.nn as nn

class SelfAttention(nn.Module):
    """单头 self-attention：X -> (W_Q,W_K,W_V 投影) -> scaled dot-product attention。
    这是第 7 章多头注意力的「单头」基石。"""
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)   # X -> Q
        self.W_K = nn.Linear(d_model, d_k, bias=False)   # X -> K
        self.W_V = nn.Linear(d_model, d_k, bias=False)   # X -> V

    def forward(self, X, causal=False):
        # X: [B, L, d_model]
        Q, K, V = self.W_Q(X), self.W_K(X), self.W_V(X)  # 各 [B, L, d_k]
        mask = None
        if causal:
            L = X.size(1)
            mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device=X.device),
                              diagonal=1)                # [L, L] 自动广播到 batch
        out, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
        return out, attn

torch.manual_seed(0)
B, L, d_model, d_k = 1, 5, 32, 16
X = torch.randn(B, L, d_model)                           # 一条长度 5 的 toy 序列
sa = SelfAttention(d_model, d_k)
out, attn = sa(X, causal=True)                           # 带 causal mask 跑一遍
print("输入  X 形状:", tuple(X.shape), " (B, L, d_model)")
print("输出 out 形状:", tuple(out.shape), " (B, L, d_k)——序列长度不变，每个 token 被重表示")
print("注意力权重 attn 形状:", tuple(attn.shape), " (B, L, L)")
print("第一个样本的注意力矩阵（causal，应为下三角）:\n", attn[0].round(decimals=2))
print("→ 这就是 Transformer 里一个『头』的完整运算；第 7 章把它并排跑多份就是多头注意力。")